### เปรียบเทียบประสิทธิภาพ LLM ในการตรวจจับ Scam (Qwen, OpenAI, Gemini)

In [4]:
import os
import pandas as pd
from openrouter import OpenRouter
from dotenv import load_dotenv  

load_dotenv()
api_key=os.getenv("OPENROUTER_API_KEY")

In [1]:
SYSTEM_PROMPT = """
Role: คุณเป็นผู้เชี่ยวชาญด้านการวิเคราะห์ SMS หลอกลวง ในประเทศไทย และเป็นหน่วยรักษาความปลอดภัยทางไซเบอร์มานานกว่า 50 ปี

Task: ช้อความต่อไปนี้เป็นข้อความที่ส่งมาจาก sms ซึ่งใน ข้อความเหล่านี้จะมีทั้งข้อความปกติ และ ข้อความจากแสกมเมอร์ 
หน้าที่ของคุณคือวิเคราะห์ว่า sms เป็นข้อความปกติ หรือ ข้อความจากแสกมเมอร์พร้อมให้เหตุผล แล้วสรุปผลออกมาสั้นๆ
"""

def qwen_analysis (text):
    with OpenRouter(api_key=api_key) as client:
        response = client.chat.send(
            model="qwen/qwen3-32b",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text},
            ]
        )

    return response.choices[0].message.content
def glm_analysis (text):
    with OpenRouter(api_key=api_key) as client:
        response = client.chat.send(
            model="z-ai/glm-4.5-air",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text},
            ]
        )

    return response.choices[0].message.content

In [5]:
test_messages = [
    #Scam messages
    "ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์: https://rb.gy/2erksz",
    "คุณได้รับคืนเงินประกันมิเตอร์ไฟฟ้าส่วนภูมิภาค PEA ซึ่งจะคืนตามขนาดของมิเตอร์ที่ใช้ สามารถสอบถามเพิ่มเติมได้ที่ https://bit.ly/44jHSpi",
    "KMA ยื่นสิทธิ์การกู้ให้ท่าน 120,000 บาท คลิก: citly.me/eq9br ",
    "เงินเข้าโทรศัพท์ฟรี 500บาท กด*158# ลุ้นก่อนใคร สมัครดวงดีดวงเฮง 3บ/ขค",
    "รับฟรี 129 บาท ฝากแรกรับ 1599 บาท",
    #Normal messages
    "AIS ยกขบวนมือถือรุ่นเด็ด พร้อมโปรโมชันส่วนลดจัดเต็มในงาน Thailand Mobile Expo ณ ศูนย์การประชุมแห่งชาติสิริกิติ์ ระหว่างวันที่ 8-11 กุมภาพันธ์ 2567 เตรียมมาชอปมือถือใหม่ รับปีมังกร พร้อมลดภาษีได้เต็ม Max (ขึ้นอยู่กับอัตราเงินได้สุทธิที่เสียภาษี) แล้วพบกันภายในงานตั้งแต่ 10.00 น. – 20.00 น.",
    "OTP 930917 (ref. 8697) สำหรับสมัคร myID อย่าบอกรหัสนี้กับบุคคลอื่น",
    "ใจฟู เน็ตฟูลสปีด! เล่นไม่สะดุด 50วัน 199บ. 30วัน ถึง 29 ก.พ.67 คลิก https://dtac.co.th/s/CMTDTE5",
    "[ ⬛️ Express] พัสดุ ⬛️⬛️ จัดส่งสำเร็จแล้ว ตรวจสอบได้ที่เว็บไซต์ ⬛️ Express Thailandหากมีข้อสงสัย กรุณาติดต่อ call center 1470",
    "มีผู้เข้าสู่ระบบธนาคารของคุณจากอุปกรณ์อื่น หากไม่ได้ดำเนินการด้วยตนเอง โปรดติดต่อทันที kasikorn.go-line.cc"
    ]

results = []
for msg in test_messages:
    results.append({
        "Message": msg,
        "qwen3-32b ": qwen_analysis(msg),
        "glm-4.5-air": glm_analysis(msg)
    })

df_results = pd.DataFrame(results)
display(df_results)

,Message,qwen3-32b,glm-4.5-air
0,"ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์...",**วิเคราะห์ SMS:** \nข้อความนี้มีลักษณะคล้ายข...,### ผลการวิเคราะห์ SMS:\n**เป็นข้อความจากแสกมเ...
1,คุณได้รับคืนเงินประกันมิเตอร์ไฟฟ้าส่วนภูมิภาค ...,"**วิเคราะห์ SMS ข้อความที่ได้รับ** \nคำว่า ""ค...",**การวิเคราะห์ SMS**\n\nหมายเหตุ: ข้อความนี้เป...
2,"KMA ยื่นสิทธิ์การกู้ให้ท่าน 120,000 บาท คลิก: ...",ข้อความนี้เป็น **ข้อความหลอกลวง (Scam SMS)** เ...,"### วิเคราะห์ข้อความ SMS:\n**ข้อความ:** ""KMA ย..."
3,เงินเข้าโทรศัพท์ฟรี 500บาท กด*158# ลุ้นก่อนใคร...,**วิเคราะห์ว่าเป็นข้อความปกติหรือแสกมเมอร์:** ...,**การวิเคราะห์ข้อความนี้:**\n\nข้อความนี้เป็น ...
4,รับฟรี 129 บาท ฝากแรกรับ 1599 บาท,"**วิเคราะห์ SMS:** \nข้อความ ""รับฟรี 129 บาท ...",**ผลการวิเคราะห์: ข้อความหลอกลวง (Scam)**\n\n*...
5,AIS ยกขบวนมือถือรุ่นเด็ด พร้อมโปรโมชันส่วนลดจั...,**บทสรุป:** \nข้อความข้างต้นเป็น **ข้อความปกต...,### การวิเคราะห์ข้อความ SMS\n\n**ประเภทข้อความ...
6,OTP 930917 (ref. 8697) สำหรับสมัคร myID อย่าบอ...,**วิเคราะห์ SMS:** \nข้อความนี้เป็นข้อความปกต...,**การวิเคราะห์ (Analysis):**\n\nข้อความนี้มีคว...
7,ใจฟู เน็ตฟูลสปีด! เล่นไม่สะดุด 50วัน 199บ. 30ว...,**วิเคราะห์**: ข้อความนี้มีลักษณะคล้ายข่าวร้าย...,**ผลการวิเคราะห์: ข้อความปกติ**\n\n**เหตุผล:**...
8,[ ⬛️ Express] พัสดุ ⬛️⬛️ จัดส่งสำเร็จแล้ว ตรวจ...,**วิเคราะห์**: \nข้อความนี้มีลักษณะคล้ายการหล...,### ผลการวิเคราะห์ SMS: เป็น **ข้อความหลอกลวง ...
9,มีผู้เข้าสู่ระบบธนาคารของคุณจากอุปกรณ์อื่น หาก...,**ผลการวิเคราะห์ SMS:** ข้อความนี้เป็น **ข้อคว...,**วิเคราะห์ข้อความ SMS:** \n**ข้อความนี้เป็นข...


In [4]:
pd.set_option('display.max_colwidth', None)

df_results = pd.DataFrame(results)
display(df_results)

,Message,qwen3-32b,glm-4.5-air
0,"ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์: https://rb.gy/2erksz","**วิเคราะห์ SMS:** \nข้อความเริ่มต้นด้วยข้อความเชิงล่อลวงว่า ""ธนาคารได้อนุมัติวงเงิน50,000บาท"" ซึ่งมักเป็นเทคนิคมาตรฐานของแสกมเมอร์ที่ใช้สร้างความน่าสนใจโดยไม่ต้องผู้รับอนุมัติล่วงหน้า \n- **ลิงก์ที่น่าสงสัย**: ลิงก์ที่สั้น (`https://rb.gy/2erksz`) และไม่เชื่อมโยงกับโดเมนธนาคารหลักในไทย (เช่น `scb.co.th`, `kbank.co.th`) จัดเป็นลิงก์ที่แสกมเมอร์ตั้งขึ้นเพื่อกำหนดเป้าหมาย \n- **การใช้ภาษาและไวยากรณ์ผิดรูปแบบ**: คำว่า ""ติดต่อรับสิทธิ์"" สะท้อนการออกแบบข้อความด้วยเครื่องมือแปลภาษาอัตโนมัติ ซึ่งไม่เป็นไปตามข้อความทางการของธนาคาร \n- **การสร้างอารมณ์สู่การคลิกทันที**: แสกมเมอร์ใช้คำกระตุ้นเช่น ""รีบติดต่อ"" (แม้จะไม่เขียนครบถ้วนในข้อความนี้) เพื่อหลอกให้เห็นผลเร็วโดยไม่ตรวจสอบรายละเอียด \n\n**เหตุผลจัดเป็นข้อความจากแสกมเมอร์** \n1. **ลิงก์ปลอม**: ลิงก์ที่เชื่อมโยงไปยังไซต์ที่ไม่ปลอดภัย อาจนำผู้ใช้งานไปยังหน้ารวบรวมข้อมูลหรือดาวน์โหลดมัลแวร์ \n2. **การหลอกให้คลิกโดยไม่ยืนยันตัวตน**: ไม่ปรากฏข้อมูลติดต่อหรือชื่อบริษัทที่ชัดเจน \n3. **ใช้ประโยชน์จากความไว้วางใจในธนาคาร**: แอบอ้างชื่อบริษัทเพื่อลดการตั้งคำถามจากผู้รับ \n\n**สรุปผล** \n✅ **ข้อยืนยัน**: ข้อความนี้เป็น SMS แสกมที่ปลอมตัวเป็นธนาคารและหลอกให้คลิกลิงก์เพื่อนำทางไปยังเป้าหมายอันตราย \n❌ ไม่ควรกดลิงก์ดังกล่าว ให้ตรวจสอบกับธนาคารผ่านช่องทางหลัก (เช่น เว็บไซต์หรือแอปพลิเคชันที่เป็นทางการ) หรือโทรไปสายด่วนของธนาคารโดยตรง","## การวิเคราะห์ SMS: ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์\n\n### ผลการวิเคราะห์: **ข้อความจากแสกมเมอร์ (Scam Message)**\n\n### เหตุผลที่บ่งชี้ว่าเป็นแสกมเมอร์:\n\n1. **การอ้างถึงสิทธิพิเศษที่น่าสงสัย**: \n - การอนุมัติวงเงิน 50,000 บาทโดยไม่ผ่านกระบวนการพิจารณาจริงอย่างที่ธนาคารปกตีทำ\n - การเสนอสิทธิ์ที่ดีเกินไปจริงในไม่มีเงื่อนไขหรือหลักเกณฑ์การให้บริการ\n\n2. **ลิงก์แปลกปลอม**:\n - URL ""https://rb.gy/2erksz"" ไม่ใช่เว็บไซต์ทางการของธนาคารใดๆ\n - ลิงก์สั้นแบบนี้มักนำไปยังเว็บไซต์หลอกลวงที่จะขโมยข้อมูลส่วนตัวหรงินฝาก\n\n3. **การไม่ระบุชื่อธนาคารชัดเจน**:\n - ข้อความไม่ได้ระบุว่าเป็นธนาคารอะไรที่แน่นอน (เช่น ธนาคารกรุงไทย, ธนาคารกรุงเทพ, ธนาคารไทยพาณิชย์)\n\n4. **การกระตุ้นให้กระทำทันที**:\n - ใช้คำว่า ""ติดต่อรับสิทธิ์"" เพื่อสร้างความเร่งรีบและเข้าใจผิดว่าเป๋าหน้าที่จำกัดเวลา\n\n5. **รูปแบบข้อความไม่เป็นทางการ**:\n - ข้อความสั้นๆ ไม่มีข้อมูลติดต่อหรือข้อมูลเพิ่มเติมตามมาตรฐานของธนาคาร\n\n### ข้อควรระวัง:\n- **ไม่คลิกลิงก์ในข้อความ SMS ที่ไม่ได้รับรอง**\n- **ตรวจสอบกับธนาคารโดยตรงผ่านช่องทางที่ปลอดภัย** (เช่น แอพธนาคาร, โทรศัพท์จริง, สาขาธนาคาร)\n- **ไม่กรอกข้อมูลส่วนตัว/ข้อมูลการเงินผ่านลิงก์ที่ไม่น่าเชื่อถือ**\n\n### แนวทางป้องกัน:\n- ติดตั้งแอปธนาคารทางการของทุกธนาคารที่คุณใช้งาน\n- เปิดใช้งานการแจ้งเตือนธนาคารผ่าน SMS หรืออีเมลที่ได้ลงทะเบียนไว้\n- รายงานข้อความหลอกลวงผ่านช่องทางอย่างเป็นทางการ (เช่น โทร 1326 ของ คอมพิวเตอร์คราวฟันดิ้ง)\n\nข้อความนี้เป็นตัวอย่างคลาสสิกของแสกมเมอร์ที่เลียนแบบการส่งข้อความจากธนาคารเพื่อล่อลวงให้ผู้รับคลิกลิงก์ปลอมและขโมยข้อมูลการเงิน"
1,คุณได้รับคืนเงินประกันมิเตอร์ไฟฟ้าส่วนภูมิภาค PEA ซึ่งจะคืนตามขนาดของมิเตอร์ที่ใช้ สามารถสอบถามเพิ่มเติมได้ที่ https://bit.ly/44jHSpi,"**ผลการวิเคราะห์ SMS: ข้อความนี้เป็น **""ข้อความจากแสกมเมอร์"" (Scam)** \n**เหตุผล:**\n\n1. **ใช้ชื่อบริษัทที่น่าเชื่อถือเพื่อลวงความสนใจ:** \n - ใช้ชื่อ ""มิเตอร์ไฟฟ้าส่วนภูมิภาค PEA"" ซึ่งเป็นผู้ให้บริการไฟฟ้าในพื้นที่จริง และเป็นชื่อที่ผู้ใช้รู้จักดีในประเทศไทย\n\n2. **มีข้อเสนอเงินคืนโดยไม่ชัดเจน:** \n - ข้อความกล่าวถึง ""คุณได้รับคืนเงินประกัน"" โดยไม่ได้ระบุเหตุผลหรือข้อกำหนดชัดเจน ซึ่งเป็นวิธีการแสกมที่มักใช้เพื่อดึงความสนใจให้คลิกลิงก์.\n\n3. **ใช้ลิงก์อัญเชิญ (Bitly Short Link):**\n - การใช้ Link แบบ short link (เช่น ""https://bit.ly/44jHSpi"") เป็นการปกปิดปลายทางเชิงลึกของลิงก์ ซึ่งอาจนำไปสู่เว็บไซต์ปลอมเพื่อขโมยข้อมูลส่วนตัว เช่น ชื่อผู้ใช้ เลขบัญชีธนาคาร หรือรหัส OTP\n\n4. **ไม่ระบุชื่อหรือเบอร์ผู้ส่งอย่างแน่ชัด:** \n - ข้อความไม่ระบุชื่อผู้ส่งอย่างเป็นทางการจาก PEA ไม่มีเบอร์ติดต่ออย่างชัดเจน ซึ่งเป็นลักษณะแปลกสำหรับข้อมูลจากรัฐหรือบริษัทที่มีชื่อเสียง\n\n---\n\n**สรุป:**\nSMS นี้มีลักษณะการหลอกลวง (Scam) อย่างชัดเจน ไม่เคยมีนโยบาย